# 4.2 — Prepare and Manage Documents

**Exam domain:** Domain 4.0 — Document Processing · **Weight:** 15%

## The problem this solves

The parsing call is one line. Everything that goes wrong happens before it. The file is in the
bucket but the query says it is not there. The analyst can run the function but not read the stage.
Someone shares a document link in a ticket and it still works three weeks later. None of these are
AI problems — they are file-plumbing problems, and they account for most of the time people lose on
document pipelines.

This notebook is about getting the document to the function: where it lives, who may read it, and
which of the three ways to point at a file is the right one.

## What you will be able to do

- Create a stage with a directory table and keep that catalogue in step with reality
- Grant exactly the privileges a document pipeline needs, and name the failure each missing one causes
- Choose between `TO_FILE`, `BUILD_SCOPED_FILE_URL` and `GET_PRESIGNED_URL` and say what each exposes
- Screen files against the documented size and format limits before you spend credits on them
- Diagnose a "file not found" that is not about the file existing

## Before you start

- Run `setup/dataset.sql`
- Have the `sample_docs/` files to hand — this notebook uploads them
- You need a role that can `CREATE STAGE` in `GENAI_STUDY.PUBLIC`, and ACCOUNTADMIN for the grants

📖 **Snowflake documentation for this notebook**
- [Parsing documents with AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)
- [Directory tables](https://docs.snowflake.com/en/user-guide/data-load-dirtables)
- [Querying a directory table](https://docs.snowflake.com/en/user-guide/data-load-dirtables-query)
- [GET_PRESIGNED_URL](https://docs.snowflake.com/en/sql-reference/functions/get_presigned_url)
- [BUILD_SCOPED_FILE_URL](https://docs.snowflake.com/en/sql-reference/functions/build_scoped_file_url)
- [FILE data type](https://docs.snowflake.com/en/sql-reference/data-types-unstructured)
- [AI function privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


---
## A stage is a location; a directory table is the catalogue

A **stage** is a named pointer to storage — either space Snowflake manages for you (internal) or a
bucket of yours registered through a storage integration (external). Files on a stage are reachable,
not queryable.

A **directory table** is an optional catalogue attached to that stage. Turn it on with
`DIRECTORY = (ENABLE = TRUE)` and `DIRECTORY(@my_stage)` behaves like a table:

| Column | Type | Contents |
|---|---|---|
| `RELATIVE_PATH` | TEXT | Path to the file within the stage |
| `SIZE` | NUMBER | File size in bytes |
| `LAST_MODIFIED` | TIMESTAMP_TZ | When the file was last updated on the stage |
| `MD5` | HEX | MD5 checksum |
| `ETAG` | HEX | ETag header |
| `FILE_URL` | TEXT | Snowflake file URL |

The catalogue is metadata, and metadata does not update itself when something writes to the bucket
behind Snowflake's back. `ALTER STAGE … REFRESH` reconciles it; `AUTO_REFRESH = TRUE` on an external
stage wires up cloud event notifications so it reconciles itself. Auto-refresh is not free — the
overhead of managing those notifications appears on your bill as Snowpipe charges, and a manual
refresh is billed as cloud services.

→ [More on directory tables](https://docs.snowflake.com/en/user-guide/data-load-dirtables)

### What `AI_PARSE_DOCUMENT` will accept

| Format | Notes |
|---|---|
| PDF, DOCX, PPTX | The paged formats — these are the three that support `page_split` |
| JPEG, JPG, PNG | One file bills as one page |
| TIFF, TIF | Multi-page TIFF is supported |
| HTML, TXT | Billed per 3,000-character chunk |

Spreadsheets are not on that list. `AI_EXTRACT` accepts a wider set again, adding PPT, DOC, EML,
BMP, GIF, WEBP and MD.

| Limit | `AI_PARSE_DOCUMENT` | `AI_EXTRACT` with a file |
|---|---|---|
| Max file size | 100 MB | 100 MB |
| Max pages | 2,000 | 125 (fewer with `scale_factor` above 1.0) |
| Max resolution | 10,000 × 10,000 px | — |
| Page selection | `page_filter`, array of 0-based `{start, end}` | — |

→ [More on formats and limits](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)


In [ ]:
%%sql
-- Create an internal stage for document uploads
CREATE STAGE IF NOT EXISTS GENAI_STUDY.PUBLIC.DOCS_STAGE
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Upload PDF and image files here for AI_PARSE_DOCUMENT exercises';


In [ ]:
%%sql -r stage_create_2
-- Verify stage creation
SHOW STAGES IN SCHEMA GENAI_STUDY.PUBLIC;


### The privileges, and the error each missing one produces

```sql
GRANT USE AI FUNCTIONS ON ACCOUNT              TO ROLE <role>;   -- or USE AI FUNCTION <name> ON ACCOUNT
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER      TO ROLE <role>;   -- or SNOWFLAKE.AI_FUNCTIONS_USER
GRANT USAGE  ON DATABASE <db>                  TO ROLE <role>;
GRANT USAGE  ON SCHEMA   <schema>              TO ROLE <role>;
GRANT READ   ON STAGE    <internal_stage>      TO ROLE <role>;   -- USAGE for an external stage
```

You need **both** halves of the AI grant: the account-level privilege *and* one of the two database
roles. `USE AI FUNCTIONS` and `SNOWFLAKE.CORTEX_USER` are granted to `PUBLIC` by default, so on a
fresh account this often appears to work without you doing anything — and then fails on the
hardened account where someone revoked the `PUBLIC` grant.

`SNOWFLAKE.AI_FUNCTIONS_USER` is the narrower of the two roles: scalar functions only, excluding the
aggregate ones such as `AI_AGG` and `AI_SUMMARIZE_AGG`. It is not granted to `PUBLIC`. Only
ACCOUNTADMIN can grant or revoke any of this.

→ [More on AI function privileges](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


In [ ]:
%%sql
-- Grant read access to the analyst role
GRANT READ ON STAGE GENAI_STUDY.PUBLIC.DOCS_STAGE TO ROLE GENAI_ANALYST;


In [ ]:
%%sql -r list_stage_files_1
-- The directory table as a queryable catalogue.
-- Requires DIRECTORY = (ENABLE = TRUE) on the stage; an empty result usually means
-- the metadata has not been refreshed, not that the stage is empty.
SELECT
    RELATIVE_PATH,
    SIZE                       AS file_size_bytes,
    ROUND(SIZE / 1048576, 2)   AS file_size_mb,
    LAST_MODIFIED,
    FILE_URL
FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE)
ORDER BY LAST_MODIFIED DESC;


In [ ]:
%%sql
-- Refresh directory metadata (needed after new uploads)
ALTER STAGE GENAI_STUDY.PUBLIC.DOCS_STAGE REFRESH;


> ### ⚠️ Common misconceptions
>
> **"The file is in the bucket, so the query can see it."**
> A directory table is cached metadata. Files written by anything other than Snowflake are invisible
> until `ALTER STAGE … REFRESH` runs, or until an `AUTO_REFRESH` notification arrives. The symptom
> is a `DIRECTORY()` query that returns nothing and an `AI_PARSE_DOCUMENT` call that reports the
> file is not found — while you are looking at the object in the AWS console.
> → [Directory tables](https://docs.snowflake.com/en/user-guide/data-load-dirtables)
>
> **"Any stage works, it's all just storage."**
> AI functions cannot build a FILE object over a **user stage** (`@~`), a **table stage** (`@%tbl`),
> an internal stage with `TYPE = 'SNOWFLAKE_FULL'` encryption, an external stage using customer-side
> encryption such as `AWS_CSE` or `AZURE_CSE`, or a stage whose name is double-quoted. The failure
> reads like an access or not-found error, which sends people hunting for a missing grant that is
> already there.
> → [FILE limitations for AI functions](https://docs.snowflake.com/en/sql-reference/functions/ai_embed)
>
> **"I granted `SNOWFLAKE.CORTEX_USER`, so the role can call AI functions."**
> The database role is one of two requirements. Without the `USE AI FUNCTIONS` account privilege
> (or a per-function `USE AI FUNCTION <name>` grant) the call is refused. It usually looks fine in
> development because both are granted to `PUBLIC` by default.
> → [AI function privileges](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


In [ ]:
%%sql -r list_stage_files_3
-- Verify file count
SELECT COUNT(*) AS file_count FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE);


---
## Three ways to point at a staged file

They are not interchangeable, and the difference is who can use the result.

| | What it is | Who can use it | Lifetime |
|---|---|---|---|
| `TO_FILE('@stage','path')` | A FILE object — a scalar value, not a URL | SQL, inside your session | The value itself; it may go stale if the underlying file changes |
| `BUILD_SCOPED_FILE_URL('@stage','path')` | An encoded, scoped URL of the form `https://<account>/api/files/<query_id>/<encoded_path>` | The caller, authenticated to Snowflake | Valid until the persisted query result period ends — currently **24 hours** |
| `GET_PRESIGNED_URL('@stage','path'[, secs])` | A pre-signed storage URL | **Anyone holding the string.** No Snowflake login required | Default **3,600 s**; maximum depends on the storage |

`GET_PRESIGNED_URL` maximums: 3,600 seconds for AWS S3 accessed through an IAM role and for
Microsoft Fabric OneLake; 604,800 seconds (7 days) for other external stages. It requires
server-side encryption on the stage, and it will hand you a URL even when the file does not exist —
so a 403 or 404 from the browser is not evidence about your privileges.

Privileges: `USAGE` on an external stage, `READ` on an internal one, for both URL functions.

A pre-signed URL is a bearer credential. Pasting one into a ticket, a chat thread or a dashboard
gives everyone who reads that thread the document, for as long as the token lives. Treat the
expiry argument as a security control, not a convenience knob.

→ [More on scoped file URLs](https://docs.snowflake.com/en/sql-reference/functions/build_scoped_file_url) ·
→ [More on pre-signed URLs](https://docs.snowflake.com/en/sql-reference/functions/get_presigned_url)


In [ ]:
%%sql -r file_urls_1
-- ============================================================
-- 1. TO_FILE -- the FILE object AI functions consume. Not a URL.
--    Carries RELATIVE_PATH, STAGE, SIZE, ETAG, LAST_MODIFIED, CONTENT_TYPE.
-- ============================================================
SELECT TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_KF-2041.pdf') AS file_ref;


In [ ]:
%%sql -r file_urls_2
-- 2. BUILD_SCOPED_FILE_URL -- an encoded, scoped URL for use inside Snowflake by the caller.
--    Valid until the persisted query result period ends (currently 24 hours).
SELECT
    RELATIVE_PATH,
    BUILD_SCOPED_FILE_URL('@GENAI_STUDY.PUBLIC.DOCS_STAGE', RELATIVE_PATH) AS scoped_url
FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE)
WHERE LOWER(RELATIVE_PATH) LIKE '%.pdf';


In [ ]:
%%sql -r file_urls_3
-- 3. GET_PRESIGNED_URL -- usable OUTSIDE Snowflake, with no Snowflake authentication.
--    GET_PRESIGNED_URL( @<stage>, '<relative_path>' [, <expiration_seconds> ] )
--    Default expiry: 3600 seconds. Maximum depends on the storage:
--       AWS S3 accessed via an IAM role  -> 3,600 s
--       Microsoft Fabric OneLake         -> 3,600 s
--       other external stages            -> 604,800 s (7 days)
--    Requires server-side encryption on the stage.
--    Privilege: USAGE on an external stage, READ on an internal stage.
SELECT GET_PRESIGNED_URL('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'contract_msa_harbourview.pdf', 3600)
       AS presigned_url;

-- Anyone holding this string can read the contract until it expires. Shorten the expiry
-- rather than lengthen it, and never paste one into a shared channel.


> ### ⚠️ Common misconceptions
>
> **"`BUILD_SCOPED_FILE_URL` gives a permanent internal link."**
> It is scoped and time-limited: valid for the caller until the persisted query result period ends,
> currently 24 hours. Store one in a table as a long-lived reference and it will work all through
> testing and be dead by the time the report runs tomorrow.
> → [BUILD_SCOPED_FILE_URL](https://docs.snowflake.com/en/sql-reference/functions/build_scoped_file_url)
>
> **"A pre-signed URL respects the viewer's Snowflake role."**
> It respects nothing. It is a storage-level token that works for whoever holds it, with no
> Snowflake authentication at all. Row access policies, masking and role hierarchy do not apply to
> the bytes behind that link.
> → [GET_PRESIGNED_URL](https://docs.snowflake.com/en/sql-reference/functions/get_presigned_url)
>
> **"If `GET_PRESIGNED_URL` returns a URL, the file must exist."**
> The function generates a URL even when the file is not on the stage. Use the directory table to
> check existence; use the URL only to fetch.
> → [GET_PRESIGNED_URL](https://docs.snowflake.com/en/sql-reference/functions/get_presigned_url)


In [ ]:
%%sql -r file_validation
-- Pre-flight: screen the stage against the documented limits before spending credits.
SELECT
    RELATIVE_PATH,
    ROUND(SIZE / 1048576.0, 2) AS file_mb,
    CASE
        WHEN SIZE > 104857600 THEN 'EXCEEDS_100MB_LIMIT'
        WHEN LOWER(RELATIVE_PATH) RLIKE '.*\\.(pdf|docx|pptx)$'          THEN 'PAGED_DOC_OK'
        WHEN LOWER(RELATIVE_PATH) RLIKE '.*\\.(jpg|jpeg|png|tif|tiff)$'  THEN 'IMAGE_OK'
        WHEN LOWER(RELATIVE_PATH) RLIKE '.*\\.(html|htm|txt)$'           THEN 'TEXT_OK'
        WHEN LOWER(RELATIVE_PATH) RLIKE '.*\\.(xlsx|csv|mp4)$'           THEN 'UNSUPPORTED_FOR_PARSE_DOCUMENT'
        ELSE 'UNKNOWN'
    END AS parse_readiness,
    CASE
        WHEN LOWER(RELATIVE_PATH) RLIKE '.*\\.(pdf|docx|pptx)$'          THEN 'LAYOUT or OCR; page_split supported'
        WHEN LOWER(RELATIVE_PATH) RLIKE '.*\\.(jpg|jpeg|png|tif|tiff)$'  THEN 'OCR or LAYOUT; 1 file = 1 billed page'
        WHEN LOWER(RELATIVE_PATH) RLIKE '.*\\.(html|htm|txt)$'           THEN 'billed per 3,000-character chunk'
        ELSE 'N/A'
    END AS notes
FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE);

-- For a document that is too long rather than too large, narrow the pages instead of rejecting it:
--   AI_PARSE_DOCUMENT(TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE','contract_msa_harbourview.pdf'),
--                     {'mode':'OCR','page_filter':[{'start':0,'end':2}]})


> ### 🤔 Stop and think
>
> - Legal asks for a link to `contract_msa_harbourview.pdf` they can open from a phone, outside the
>   VPN. `GET_PRESIGNED_URL` does it in one line. What have you just moved outside every governance
>   control the warehouse offers, and what would you need in place before you would agree?
> - `AUTO_REFRESH = TRUE` removes a class of bug and adds a line to the bill under Snowpipe. At what
>   document volume does a nightly `ALTER STAGE … REFRESH` become the wrong answer?
> - Pre-flight validation keeps unsupported files out of a batch — and quietly drops them. Who finds
>   out that the 140 MB scan was never processed, and how?


In [ ]:
# Upload a document from the notebook workspace to the stage using the Snowpark file API.
# PUT works from SnowSQL and the Snowsight stage UI too; this is the programmatic route.

from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Point this at the sample document you want to stage. The text ticket file is the smallest
# of the fixtures and is the one used for the text-format billing examples (3,000-character chunks).
LOCAL_PATH = 'sample_docs/support_tickets.txt'

session.file.put(
    local_file_name=LOCAL_PATH,
    stage_location='@GENAI_STUDY.PUBLIC.DOCS_STAGE',
    auto_compress=False,      # leave this FALSE: a .gz is not a format AI_PARSE_DOCUMENT accepts
    overwrite=True
)

# The directory table is metadata and does not update itself on PUT from every client path.
session.sql('ALTER STAGE GENAI_STUDY.PUBLIC.DOCS_STAGE REFRESH').collect()

print(session.sql(
    "SELECT RELATIVE_PATH, SIZE FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE) ORDER BY RELATIVE_PATH"
).collect())


---
## External stages

```sql
CREATE STAGE GENAI_STUDY.PUBLIC.S3_DOCS
    URL = 's3://my-company-docs/invoices/'
    STORAGE_INTEGRATION = MY_S3_INT
    DIRECTORY = (ENABLE = TRUE
                 AUTO_REFRESH = TRUE);   -- cloud event notifications keep the catalogue current
```

How it differs from an internal stage:

- The bytes stay in S3, Azure Blob or GCS. Snowflake reads them in place; nothing is copied in.
- `AUTO_REFRESH = TRUE` subscribes to cloud notifications (SQS/SNS, Event Grid, Pub/Sub) so the
  directory table updates itself. The notification-management overhead is billed as Snowpipe.
- Authentication goes through a `STORAGE INTEGRATION` rather than Snowflake's own credentials.
- Grant `USAGE ON STAGE` for an external stage where you would grant `READ ON STAGE` for an internal
  one.

The trade-off is ownership of failure. An internal stage fails in ways Snowflake can tell you about.
An external stage fails in ways the cloud provider knows about and Snowflake reports as "not found".

---
## Scenario

**Situation.** A company stores customer contracts as PDFs on S3 behind an external stage.
`AI_PARSE_DOCUMENT` reports that the file cannot be found. The objects are visibly there in the S3
console.

**Question.** Give three plausible causes and the fix for each.

### Worked solution

**1. The directory table is stale.** Objects written directly to S3 do not appear in Snowflake's
catalogue until the metadata is reconciled. This is the first thing to check because it is the most
common and the cheapest to rule out.

```sql
ALTER STAGE my_s3_stage REFRESH;
SELECT COUNT(*) FROM DIRECTORY(@my_s3_stage);
-- permanent fix: DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE)
```

**2. The storage integration cannot read the object.** The IAM role behind the integration is
missing `s3:GetObject`, or `s3:ListBucket` on the prefix. Snowflake surfaces this as a file access
problem, not a permissions one.

```sql
DESC STORAGE INTEGRATION MY_S3_INT;   -- check STORAGE_AWS_IAM_USER_ARN and STORAGE_AWS_EXTERNAL_ID
DESC STAGE my_s3_stage;
```

**3. The path does not mean what you think.** The stage `URL` already includes a prefix, and
`RELATIVE_PATH` is relative to it — passing a full key, or a leading slash, produces a path that
resolves nowhere.

```sql
LIST @my_s3_stage;
SELECT RELATIVE_PATH FROM DIRECTORY(@my_s3_stage) LIMIT 10;
```

**Two more that produce the same message**

4. **A missing privilege.** The calling role needs `USAGE` on an external stage (`READ` on an
   internal one), plus `USAGE` on the database and schema.
5. **A stage type FILE objects cannot use.** User stages, table stages, internal stages with
   `TYPE = 'SNOWFLAKE_FULL'`, external stages with customer-side encryption (`AWS_CSE`, `AZURE_CSE`)
   and double-quoted stage names are all unsupported for the FILE objects these functions take.

Work them in that order. Steps 1 and 3 cost a single query each; step 2 costs a conversation with
whoever owns the IAM role.


---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** Name the six columns of a directory table.

<details><summary>Show answer</summary>

`RELATIVE_PATH`, `SIZE`, `LAST_MODIFIED`, `MD5`, `ETAG`, `FILE_URL`. `SIZE` is in bytes and
`LAST_MODIFIED` is a `TIMESTAMP_TZ`. There is no `FILE_NAME` column — the path is the identifier,
which matters when files live in subfolders of the stage.

→ [Querying a directory table](https://docs.snowflake.com/en/user-guide/data-load-dirtables-query)

</details>

**2.** What is the default expiry of `GET_PRESIGNED_URL`, and what is the maximum?

<details><summary>Show answer</summary>

The default is 3,600 seconds. The maximum depends on the storage: 3,600 seconds for AWS S3 accessed
through an IAM role and for Microsoft Fabric OneLake, and 604,800 seconds (7 days) for other
external stages. Asking for 7 days on an S3-with-IAM-role stage does not silently give you an hour —
it is outside what that storage permits.

→ [GET_PRESIGNED_URL](https://docs.snowflake.com/en/sql-reference/functions/get_presigned_url)

</details>

**3.** Which privilege does an internal stage need for document parsing, and which does an external
one need?

<details><summary>Show answer</summary>

`READ` on an internal stage; `USAGE` on an external stage. Granting `READ` on an external stage is
the standard mix-up. On top of the stage grant the role needs `USAGE` on the database and schema,
the `USE AI FUNCTIONS` account privilege, and one of `SNOWFLAKE.CORTEX_USER` or
`SNOWFLAKE.AI_FUNCTIONS_USER`.

→ [AI function privileges](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**4.** `DIRECTORY(@my_s3_stage)` returns zero rows. The S3 console shows 400 PDFs. What do you run
first, and why that first?

<details><summary>Show answer</summary>

`ALTER STAGE my_s3_stage REFRESH;`. The directory table is cached metadata, and objects written
outside Snowflake do not register until a refresh or an auto-refresh notification. It goes first
because it is one statement and rules out the most likely cause; IAM and path problems take longer
to investigate and produce the same symptom.

→ [Directory tables](https://docs.snowflake.com/en/user-guide/data-load-dirtables)

</details>

**5.** A colleague stores `BUILD_SCOPED_FILE_URL` output in a table so reports can link to source
documents. What happens?

<details><summary>Show answer</summary>

The links work for about a day and then stop. A scoped URL is valid for the caller until the
persisted query result period ends — currently 24 hours. Store the stage and relative path instead
and build the URL at read time, so the link is generated under the reader's own privileges.

→ [BUILD_SCOPED_FILE_URL](https://docs.snowflake.com/en/sql-reference/functions/build_scoped_file_url)

</details>

**6.** Someone puts a document on their user stage (`@~`) and calls `AI_PARSE_DOCUMENT` on it. What
happens, and what does the error look like?

<details><summary>Show answer</summary>

It fails. FILE objects for AI functions are not supported on user stages, table stages, internal
stages with `TYPE = 'SNOWFLAKE_FULL'`, external stages with customer-side encryption modes such as
`AWS_CSE` or `AZURE_CSE`, or stages with double-quoted names. The message reads as a file-access or
not-found problem, so people go looking for a missing `READ` grant they already have. Move the file
to a named stage.

→ [FILE limitations for AI functions](https://docs.snowflake.com/en/sql-reference/functions/ai_embed)

</details>

**7.** `session.file.put(..., auto_compress=True)` uploads `invoice_KF-2041.pdf` and the parse now fails.
Why?

<details><summary>Show answer</summary>

`auto_compress=True` stages the file as `invoice_KF-2041.pdf.gz`. GZIP is not in the accepted format list
(PDF, PPTX, DOCX, JPEG, JPG, PNG, TIFF, TIF, HTML, TXT), and `RELATIVE_PATH` no longer matches the
name your query is looking for either. Set `auto_compress=False` for documents.

→ [Parsing documents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)

</details>

**8.** A 140 MB scanned PDF and an 800-page contract are both on the stage. Which one can
`AI_PARSE_DOCUMENT` handle, and what do you do about the other?

<details><summary>Show answer</summary>

The 800-page contract is fine — the page limit is 2,000. The 140 MB file is over the 100 MB size
limit and has to be split before it can be parsed at all; `page_filter` does not help, because the
limit is on the file, not on the pages processed. Note that `AI_EXTRACT` would reject the 800-page
contract, since its own limit is 125 pages.

→ [Parsing documents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)

</details>

**9.** An analyst can run `AI_EXTRACT` on text but gets an access error the moment the input is a
`TO_FILE` on `DOCS_STAGE`. Which grant is missing?

<details><summary>Show answer</summary>

The stage grant — `READ ON STAGE` for an internal stage. The AI privileges are clearly in place,
because the text form of the same function works. This split is diagnostic: text input needs only
the AI grants, file input additionally needs to read the stage.

→ [AI function privileges](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**10.** Support wants to attach source documents to tickets so engineers can open them. Which URL
function, and what do you have to decide first?

<details><summary>Show answer</summary>

If the engineers are Snowflake users working inside Snowsight, `BUILD_SCOPED_FILE_URL` keeps access
inside the authentication boundary and expires within 24 hours. If they are not — the usual case for
a ticketing tool — only `GET_PRESIGNED_URL` works, and you have to accept that the link is a bearer
credential that ignores roles entirely. Decide the expiry deliberately, and decide whether contract
and invoice documents may leave the governed boundary at all before you decide the syntax.

→ [GET_PRESIGNED_URL](https://docs.snowflake.com/en/sql-reference/functions/get_presigned_url)

</details>

**11.** Your stage receives 50 documents a day from an external system. Argue for `AUTO_REFRESH`
and then argue against it.

<details><summary>Show answer</summary>

For: the catalogue is never stale, files become processable within seconds of arrival, and the
biggest category of "file not found" bug disappears. Against: it requires cloud event notifications
to be configured on the provider side, it introduces a dependency outside Snowflake that can break
quietly, and the notification-management overhead is billed as Snowpipe charges. At 50 files a day
with no latency requirement, a scheduled `ALTER STAGE … REFRESH` is simpler and cheaper; at 50,000
with a minutes-level SLA it is not.

→ [Directory tables](https://docs.snowflake.com/en/user-guide/data-load-dirtables)

</details>

**12.** The pipeline works when you run it by hand and does nothing when a task runs it on a
schedule. Where do you look, and what is the underlying rule?

<details><summary>Show answer</summary>

At the grants held by the **task owner**, not by you. A task runs with the privileges of its owner
role, so a pipeline that works interactively can fail on schedule because the owner lacks
`USE AI FUNCTIONS`, a Cortex database role, or `READ` on the stage. `EXECUTE TASK` on the account
and `USAGE` on the warehouse are needed as well. This is the bridge into notebook 4.3, where the
same rule decides whether a whole document pipeline produces rows.

→ [CREATE TASK](https://docs.snowflake.com/en/sql-reference/sql/create-task)

</details>
